# Richards-Wolf Gaussian Field: Low vs High NA

Test with NA=0.1 (low) and NA=0.9 (high)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../')

from monte_carlo.richards_wolf import RichardsWolfSimulator

plt.rcParams['figure.figsize'] = (14, 10)

In [ ]:
# Parameters
wavelength = 0.532  # microns
n_medium = 1.0
polarization = 'x'
fill_factor = 0.6  # Gaussian beam fill factor

# Two NAs to test
NA_low = 0.1
NA_high = 0.9

print(f"Wavelength: {wavelength} μm")
print(f"Fill factor: {fill_factor}")
print(f"Low NA: {NA_low}")
print(f"High NA: {NA_high}")

In [ ]:
# Create simulators
print("Creating simulators...")

# Low NA - Uniform
sim_low_uniform = RichardsWolfSimulator(
    wavelength=wavelength,
    numerical_aperture=NA_low,
    n_medium=n_medium,
    polarization=polarization,
    input_field='uniform'
)

# Low NA - Gaussian
sim_low_gaussian = RichardsWolfSimulator(
    wavelength=wavelength,
    numerical_aperture=NA_low,
    n_medium=n_medium,
    polarization=polarization,
    input_field='gaussian',
    fill_factor=fill_factor
)

# High NA - Uniform
sim_high_uniform = RichardsWolfSimulator(
    wavelength=wavelength,
    numerical_aperture=NA_high,
    n_medium=n_medium,
    polarization=polarization,
    input_field='uniform'
)

# High NA - Gaussian
sim_high_gaussian = RichardsWolfSimulator(
    wavelength=wavelength,
    numerical_aperture=NA_high,
    n_medium=n_medium,
    polarization=polarization,
    input_field='gaussian',
    fill_factor=fill_factor
)

print(f"Low NA Airy radius: {sim_low_uniform.airy_radius:.4f} μm")
print(f"High NA Airy radius: {sim_high_uniform.airy_radius:.4f} μm")
print("Done.")

In [ ]:
# Compute focal plane (z=0) intensity
print("Computing focal plane intensity patterns...")

# Grid for low NA (larger spot)
n_points = 150
x_max_low = 4.0  # microns
x_low = np.linspace(-x_max_low, x_max_low, n_points)
y_low = np.linspace(-x_max_low, x_max_low, n_points)
X_low, Y_low = np.meshgrid(x_low, y_low)

# Grid for high NA (smaller spot)
x_max_high = 1.0  # microns
x_high = np.linspace(-x_max_high, x_max_high, n_points)
y_high = np.linspace(-x_max_high, x_max_high, n_points)
X_high, Y_high = np.meshgrid(x_high, y_high)

print("  Computing Low NA uniform...")
I_low_uniform = sim_low_uniform.focal_plane_intensity_pattern(X_low.flatten(), Y_low.flatten())
I_low_uniform = I_low_uniform.reshape(X_low.shape)

print("  Computing Low NA Gaussian...")
I_low_gaussian = sim_low_gaussian.focal_plane_intensity_pattern(X_low.flatten(), Y_low.flatten())
I_low_gaussian = I_low_gaussian.reshape(X_low.shape)

print("  Computing High NA uniform...")
I_high_uniform = sim_high_uniform.focal_plane_intensity_pattern(X_high.flatten(), Y_high.flatten())
I_high_uniform = I_high_uniform.reshape(X_high.shape)

print("  Computing High NA Gaussian...")
I_high_gaussian = sim_high_gaussian.focal_plane_intensity_pattern(X_high.flatten(), Y_high.flatten())
I_high_gaussian = I_high_gaussian.reshape(X_high.shape)

print("Done.")

In [ ]:
# Plot focal plane comparisons
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Low NA
im0 = axes[0, 0].imshow(I_low_uniform, extent=[-x_max_low, x_max_low, -x_max_low, x_max_low],
                        cmap='hot', origin='lower')
axes[0, 0].set_title(f'Low NA={NA_low} - Uniform')
axes[0, 0].set_xlabel('x (μm)')
axes[0, 0].set_ylabel('y (μm)')
circle = plt.Circle((0, 0), sim_low_uniform.airy_radius, color='cyan', fill=False, ls=':', lw=1.5)
axes[0, 0].add_patch(circle)
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

im1 = axes[0, 1].imshow(I_low_gaussian, extent=[-x_max_low, x_max_low, -x_max_low, x_max_low],
                        cmap='hot', origin='lower')
axes[0, 1].set_title(f'Low NA={NA_low} - Gaussian (fill={fill_factor})')
axes[0, 1].set_xlabel('x (μm)')
axes[0, 1].set_ylabel('y (μm)')
circle = plt.Circle((0, 0), sim_low_gaussian.airy_radius, color='cyan', fill=False, ls=':', lw=1.5)
axes[0, 1].add_patch(circle)
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

diff_low = I_low_uniform - I_low_gaussian
vmax_low = np.max(np.abs(diff_low))
im2 = axes[0, 2].imshow(diff_low, extent=[-x_max_low, x_max_low, -x_max_low, x_max_low],
                        cmap='RdBu_r', origin='lower', vmin=-vmax_low, vmax=vmax_low)
axes[0, 2].set_title('Difference (Uniform - Gaussian)')
axes[0, 2].set_xlabel('x (μm)')
axes[0, 2].set_ylabel('y (μm)')
plt.colorbar(im2, ax=axes[0, 2], fraction=0.046, pad=0.04)

# High NA
im3 = axes[1, 0].imshow(I_high_uniform, extent=[-x_max_high, x_max_high, -x_max_high, x_max_high],
                        cmap='hot', origin='lower')
axes[1, 0].set_title(f'High NA={NA_high} - Uniform')
axes[1, 0].set_xlabel('x (μm)')
axes[1, 0].set_ylabel('y (μm)')
circle = plt.Circle((0, 0), sim_high_uniform.airy_radius, color='cyan', fill=False, ls=':', lw=1.5)
axes[1, 0].add_patch(circle)
plt.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

im4 = axes[1, 1].imshow(I_high_gaussian, extent=[-x_max_high, x_max_high, -x_max_high, x_max_high],
                        cmap='hot', origin='lower')
axes[1, 1].set_title(f'High NA={NA_high} - Gaussian (fill={fill_factor})')
axes[1, 1].set_xlabel('x (μm)')
axes[1, 1].set_ylabel('y (μm)')
circle = plt.Circle((0, 0), sim_high_gaussian.airy_radius, color='cyan', fill=False, ls=':', lw=1.5)
axes[1, 1].add_patch(circle)
plt.colorbar(im4, ax=axes[1, 1], fraction=0.046, pad=0.04)

diff_high = I_high_uniform - I_high_gaussian
vmax_high = np.max(np.abs(diff_high))
im5 = axes[1, 2].imshow(diff_high, extent=[-x_max_high, x_max_high, -x_max_high, x_max_high],
                        cmap='RdBu_r', origin='lower', vmin=-vmax_high, vmax=vmax_high)
axes[1, 2].set_title('Difference (Uniform - Gaussian)')
axes[1, 2].set_xlabel('x (μm)')
axes[1, 2].set_ylabel('y (μm)')
plt.colorbar(im5, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('../data/rw_gaussian_test.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Radial profiles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Low NA
center_idx = n_points // 2
profile_low_uniform = I_low_uniform[center_idx, :]
profile_low_gaussian = I_low_gaussian[center_idx, :]

axes[0].plot(x_low, profile_low_uniform, 'b-', label='Uniform', linewidth=2)
axes[0].plot(x_low, profile_low_gaussian, 'r-', label='Gaussian', linewidth=2)
axes[0].axvline(sim_low_uniform.airy_radius, color='cyan', ls=':', lw=1.5, label=f'Airy r={sim_low_uniform.airy_radius:.3f} μm')
axes[0].axvline(-sim_low_uniform.airy_radius, color='cyan', ls=':', lw=1.5)
axes[0].set_xlabel('x (μm)')
axes[0].set_ylabel('Normalized Intensity')
axes[0].set_title(f'Radial Profile (NA={NA_low})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# High NA
profile_high_uniform = I_high_uniform[center_idx, :]
profile_high_gaussian = I_high_gaussian[center_idx, :]

axes[1].plot(x_high, profile_high_uniform, 'b-', label='Uniform', linewidth=2)
axes[1].plot(x_high, profile_high_gaussian, 'r-', label='Gaussian', linewidth=2)
axes[1].axvline(sim_high_uniform.airy_radius, color='cyan', ls=':', lw=1.5, label=f'Airy r={sim_high_uniform.airy_radius:.3f} μm')
axes[1].axvline(-sim_high_uniform.airy_radius, color='cyan', ls=':', lw=1.5)
axes[1].set_xlabel('x (μm)')
axes[1].set_ylabel('Normalized Intensity')
axes[1].set_title(f'Radial Profile (NA={NA_high})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison
print("="*70)
print("QUANTITATIVE METRICS")
print("="*70)

def compute_metrics(I, x, airy_r, name):
    center_idx = I.shape[0] // 2
    profile = I[center_idx, :]
    
    peak = I.max()
    total = I.sum()
    
    # FWHM
    half_max = peak / 2
    above_half = profile > half_max
    fwhm = x[above_half][-1] - x[above_half][0] if above_half.any() else 0
    
    print(f"\n{name}:")
    print(f"  Peak intensity: {peak:.4f}")
    print(f"  FWHM: {fwhm:.4f} μm")
    print(f"  FWHM / Airy radius: {fwhm/airy_r:.4f}")
    print(f"  Total intensity: {total:.2e}")
    
    return {'peak': peak, 'fwhm': fwhm, 'total': total}

m_low_u = compute_metrics(I_low_uniform, x_low, sim_low_uniform.airy_radius, f"Low NA={NA_low} Uniform")
m_low_g = compute_metrics(I_low_gaussian, x_low, sim_low_gaussian.airy_radius, f"Low NA={NA_low} Gaussian")

m_high_u = compute_metrics(I_high_uniform, x_high, sim_high_uniform.airy_radius, f"High NA={NA_high} Uniform")
m_high_g = compute_metrics(I_high_gaussian, x_high, sim_high_gaussian.airy_radius, f"High NA={NA_high} Gaussian")

print("\n" + "="*70)
print("KEY OBSERVATIONS:")
print("="*70)
print(f"Low NA: Gaussian peak / Uniform peak = {m_low_g['peak']/m_low_u['peak']:.3f}")
print(f"Low NA: Gaussian FWHM / Uniform FWHM = {m_low_g['fwhm']/m_low_u['fwhm']:.3f}")
print(f"\nHigh NA: Gaussian peak / Uniform peak = {m_high_g['peak']/m_high_u['peak']:.3f}")
print(f"High NA: Gaussian FWHM / Uniform FWHM = {m_high_g['fwhm']/m_high_u['fwhm']:.3f}")
print("\nGaussian illumination → Higher peak, Narrower FWHM (more concentrated)")
print("="*70)